# Dense baselines (decision 0024)

Encodes the shared passages with all-MiniLM-L6-v2, paraphrase-multilingual-MiniLM-L12-v2, multilingual-e5-large
and BGE-M3, then (once the dev queries exist) writes TREC runs. All code and settings live in
`src/retrieval/dense.py`; this notebook only runs it.

**Setup on Kaggle**
1. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Add a private dataset holding `passages_v1.jsonl.gz` from `data/passages/` (the run refuses any file whose
   SHA-256 differs from the committed manifest).
3. To resume or to search after encoding: add this notebook's earlier output as input; finished chunks are
   reused, not re-encoded.
4. Run with **Save Version → Save & Run All**, so the run continues when the browser closes.

**Bring back:** download `dense_results.zip` from the output and unzip it at the repository root. Embeddings
stay in the Kaggle output (`embeddings/`).

In [ ]:
BRANCH = "main"  # the reports record the commit this clone is at
!git clone --quiet --branch {BRANCH} https://github.com/azzs11/heritagerec.git /tmp/heritagerec
%cd /tmp/heritagerec
!git rev-parse HEAD

In [ ]:
# transformers stays below 5: FlagEmbedding 1.4.2 fails to import with transformers 5.
!pip install -q "transformers==4.57.6" "sentence-transformers==5.7.0" "FlagEmbedding==1.4.2" "faiss-cpu==1.15.1" python-dotenv

In [ ]:
import glob, os, shutil, subprocess
from pathlib import Path

found = glob.glob("/kaggle/input/**/passages_v1.jsonl.gz", recursive=True)
assert len(found) == 1, f"expected one passages_v1.jsonl.gz under /kaggle/input, found {found}"
PASSAGES = found[0]
EMB = Path("/kaggle/working/embeddings")
EMB.mkdir(parents=True, exist_ok=True)

# Resume: copy finished chunks from an earlier run's output, if one is attached.
for old in glob.glob("/kaggle/input/**/embeddings/*/chunk_*", recursive=True):
    dest = EMB / Path(old).parent.name / Path(old).name
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(old, dest)
for old in glob.glob("/kaggle/input/**/embeddings/*/report.json", recursive=True):
    dest = EMB / Path(old).parent.name / "report.json"
    if not dest.exists():
        shutil.copy(old, dest)
print(PASSAGES, sorted(p.name for p in EMB.iterdir()))

In [ ]:
# One process per GPU: the three sentence-transformers models in turn on GPU 0, BGE-M3 on GPU 1.
def start(models, gpu):
    cmd = " && ".join(
        f"python -m src.retrieval.dense encode --model {m} --passages {PASSAGES} --emb {EMB}" for m in models
    )
    log = open(f"/kaggle/working/encode_gpu{gpu}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    return subprocess.Popen(cmd, shell=True, env=env, stdout=log, stderr=subprocess.STDOUT)

jobs = [start(["minilm_l6", "minilm_l12_multi", "e5_large"], 0), start(["bge_m3"], 1)]
codes = [j.wait() for j in jobs]
for gpu in (0, 1):
    print(f"--- GPU {gpu}, last lines"); print("".join(open(f"/kaggle/working/encode_gpu{gpu}.log").readlines()[-15:]))
assert codes == [0, 0], codes

In [ ]:
# Runs only once the dev queries are in the repository (decision 0021). The test set stays sealed: never point
# this at test queries before docs/plan/analysis_plan.md is committed.
QUERIES = Path("data/queries/dev_v1.tsv")
BGE_DENSE_ROW = False  # True also writes BGE-M3's dense-only ranking
if QUERIES.exists():
    for m in ["minilm_l6", "minilm_l12_multi", "e5_large", "bge_m3"]:
        extra = " --bge-dense-row" if (m == "bge_m3" and BGE_DENSE_ROW) else ""
        !python -m src.retrieval.dense search --model {m} --passages {PASSAGES} --emb {EMB} --queries {QUERIES}{extra}
else:
    print(f"{QUERIES} not found: encoding only")

In [ ]:
files = glob.glob("results/retrieval/dense_*_v1.json") + glob.glob("results/retrieval/runs/*.trec")
logs = glob.glob("/kaggle/working/encode_gpu*.log")
!cd /tmp/heritagerec && zip -q /kaggle/working/dense_results.zip {" ".join(files)}
!cd /kaggle/working && zip -q dense_results.zip {" ".join(os.path.basename(l) for l in logs)}
!unzip -l /kaggle/working/dense_results.zip